## Imports and Functions

In [1]:
# =============================================================================
# NIH CXR ReID — Patient Re-Identification Experiment
# =============================================================================
# Upper-boundary stress test for patient re-identifiability from chest X-rays
# before and after GS transformations.
#
# Task     : 1,739-class softmax classification over patient IDs
# Split    : train=5,217 (3/patient) | val=1,739 (1/patient) | test=3,082
# Repeats  : 5 per condition for statistical stability
# Conditions: raw, gs50, gs40, gs30, gs20, gs10, gs0
# =============================================================================

from __future__ import annotations

import os
import sys
import json
import math
import random
import gc
import time
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models
import torchvision.transforms.functional as TF

from sklearn.metrics import f1_score, top_k_accuracy_score
from scipy import stats

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# =============================================================================
# PATHS
# =============================================================================
PROJECT_ROOT  = Path('.')
REID_META_DIR = PROJECT_ROOT / 'data'  / 'metadata' / 'reid'
REID_NPZ_DIR  = PROJECT_ROOT / 'data'  / 'chest'    / 'reid_npz'
MODELS_DIR    = PROJECT_ROOT / 'models'/ 'nih_cxr'  / 'reid'
RESULTS_DIR   = PROJECT_ROOT / 'results'/'nih_cxr'  / 'reid'
FIG_DIR       = RESULTS_DIR  / 'figures'

for d in [MODELS_DIR, RESULTS_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RESULTS_CSV = RESULTS_DIR / 'NIH_CXR_ReID_Results.csv'

# =============================================================================
# REPRODUCIBILITY
# =============================================================================
GLOBAL_SEED = 1337

def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_global_seed(GLOBAL_SEED)

def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def now_str() -> str:
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# =============================================================================
# DEVICE
# =============================================================================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f"GPU  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("No GPU — running on CPU")

# =============================================================================
# EXPERIMENT CONFIGURATION
# =============================================================================
ARCHITECTURES   = ['resnet18', 'densenet121']
INITIALIZATIONS = ['Pretrained', 'Scratch']

# Conditions — reverse order (max drop first)
GS_LEVELS = ['raw', 'gs50', 'gs40', 'gs30', 'gs20', 'gs10', 'gs0']

GS_PERCENTAGE_MAP = {
    'raw'  : 0.0,
    'gs0'  : 0.0,
    'gs10' : 10.0,
    'gs20' : 20.0,
    'gs30' : 30.0,
    'gs40' : 40.0,
    'gs50' : 50.0,
}

NUM_REPEATS    = 5
NUM_EPOCHS     = 30
BATCH_SIZE     = 64     # larger than OASIS since 1,739 classes vs 347
LEARNING_RATE  = 5e-4
RESUME         = True   # skip already-completed runs

# Load class count
with open(REID_META_DIR / 'pid_to_label.json') as f:
    pid_to_label = {int(k): v for k, v in json.load(f).items()}

NUM_CLASSES = len(pid_to_label)

print(f"\n{'='*70}")
print(f"NIH CXR ReID — CONFIGURATION")
print(f"{'='*70}")
print(f"  Device          : {DEVICE}")
print(f"  Seed            : {GLOBAL_SEED}")
print(f"  Architectures   : {ARCHITECTURES}")
print(f"  Initializations : {INITIALIZATIONS}")
print(f"  GS levels       : {GS_LEVELS}")
print(f"  Num classes     : {NUM_CLASSES:,} patients")
print(f"  Repeats         : {NUM_REPEATS}")
print(f"  Epochs          : {NUM_EPOCHS}")
print(f"  Batch size      : {BATCH_SIZE}")
print(f"  Learning rate   : {LEARNING_RATE}")
print(f"  Resume          : {RESUME}")
print(f"  Results CSV     : {RESULTS_CSV}")
print(f"{'='*70}")

GPU  : NVIDIA L4
VRAM : 23.6 GB

NIH CXR ReID — CONFIGURATION
  Device          : cuda
  Seed            : 1337
  Architectures   : ['resnet18', 'densenet121']
  Initializations : ['Pretrained', 'Scratch']
  GS levels       : ['raw', 'gs50', 'gs40', 'gs30', 'gs20', 'gs10', 'gs0']
  Num classes     : 1,739 patients
  Repeats         : 5
  Epochs          : 30
  Batch size      : 64
  Learning rate   : 0.0005
  Resume          : True
  Results CSV     : results/nih_cxr/reid/NIH_CXR_ReID_Results.csv


In [2]:
# =============================================================================
# DATA LOADING
# =============================================================================

def load_reid_npz(
    split     : str,
    condition : str,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Load ReID NPZ for a given split and condition.

    Returns:
        images      : (N, 224, 224) float32
        labels      : (N,) int64    patient label (0–1738)
        patient_ids : (N,) int64    original patient ID
        genders     : (N,) int8     M=1 F=0
    """
    path = REID_NPZ_DIR / f'reid_{split}_{condition}.npz'
    if not path.exists():
        raise FileNotFoundError(
            f"NPZ not found: {path}\n"
            f"Run the NPZ builder cell first."
        )
    data = np.load(path)
    images      = data['images'].astype(np.float32)
    labels      = data['labels'].astype(np.int64)
    patient_ids = data['patient_ids'].astype(np.int64)
    genders     = data['genders'].astype(np.int8)
    data.close()
    return images, labels, patient_ids, genders


# ── Verify NPZ files exist ────────────────────────────────────────────────────
print("=== NPZ FILE CHECK ===")
all_ok = True
total_size = 0
for split in ['train', 'val', 'test']:
    for cond in GS_LEVELS:
        p = REID_NPZ_DIR / f'reid_{split}_{cond}.npz'
        if p.exists():
            size_mb = p.stat().st_size / 1e6
            total_size += size_mb
            print(f"  ✓ {p.name:<40} ({size_mb:.0f} MB)")
        else:
            print(f"  ✗ MISSING: {p.name}")
            all_ok = False

assert all_ok, "Some NPZ files missing — run NPZ builder first"
print(f"\n  Total size : {total_size/1e3:.2f} GB")

# ── Quick integrity check on raw splits ──────────────────────────────────────
print(f"\n=== RAW SPLIT INTEGRITY ===")
for split in ['train', 'val', 'test']:
    imgs, labs, pids, gens = load_reid_npz(split, 'raw')
    print(f"  {split:<6} : {imgs.shape}  "
          f"labels=[{labs.min()},{labs.max()}]  "
          f"patients={len(np.unique(pids)):,}  "
          f"M={(gens==1).sum():,} F={(gens==0).sum():,}  "
          f"range=[{imgs.min():.3f},{imgs.max():.3f}]")

print(f"\n✓ Data loading ready")


# =============================================================================
# DATASET CLASS
# =============================================================================

class ReIDDataset(Dataset):
    """
    Dataset for ReID classification.
    Applies per-condition mean/std normalisation.
    Optional light augmentation for training.
    """

    def __init__(
        self,
        images  : np.ndarray,   # (N, 224, 224) float32
        labels  : np.ndarray,   # (N,) int64
        mean    : float,
        std     : float,
        augment : bool = False,
    ):
        self.images  = images
        self.labels  = labels
        self.mean    = float(mean)
        self.std     = float(std)
        self.augment = augment

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, idx: int):
        img = self.images[idx].copy()   # (224, 224) float32

        if self.augment:
            # Rotation ±5°
            if random.random() > 0.5:
                angle = random.uniform(-5, 5)
                img   = TF.rotate(
                    torch.tensor(img).unsqueeze(0), angle
                ).numpy()[0]
            # Horizontal flip
            if random.random() > 0.5:
                img = np.fliplr(img).copy()
            # Brightness jitter ±20%
            if random.random() > 0.5:
                bf  = random.uniform(0.8, 1.2)
                lo, hi = img.min(), img.max()
                img = np.clip(img * bf, lo, hi)

        # Normalise
        img = (img - self.mean) / (self.std + 1e-8)

        x = torch.tensor(img).unsqueeze(0).float()    # (1, 224, 224)
        y = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return x, y


def compute_mean_std(
    train_imgs : np.ndarray,
    val_imgs   : np.ndarray,
) -> Tuple[float, float]:
    """Compute mean/std over train+val pool (same as OASIS)."""
    pool = np.concatenate([train_imgs, val_imgs], axis=0)
    return float(pool.mean()), float(pool.std())


print("✓ Dataset class ready")

=== NPZ FILE CHECK ===
  ✓ reid_train_raw.npz                       (235 MB)
  ✓ reid_train_gs50.npz                      (930 MB)
  ✓ reid_train_gs40.npz                      (928 MB)
  ✓ reid_train_gs30.npz                      (926 MB)
  ✓ reid_train_gs20.npz                      (923 MB)
  ✓ reid_train_gs10.npz                      (918 MB)
  ✓ reid_train_gs0.npz                       (908 MB)
  ✓ reid_val_raw.npz                         (78 MB)
  ✓ reid_val_gs50.npz                        (310 MB)
  ✓ reid_val_gs40.npz                        (309 MB)
  ✓ reid_val_gs30.npz                        (308 MB)
  ✓ reid_val_gs20.npz                        (308 MB)
  ✓ reid_val_gs10.npz                        (306 MB)
  ✓ reid_val_gs0.npz                         (303 MB)
  ✓ reid_test_raw.npz                        (138 MB)
  ✓ reid_test_gs50.npz                       (549 MB)
  ✓ reid_test_gs40.npz                       (548 MB)
  ✓ reid_test_gs30.npz                       (547 MB)
  ✓ re

In [3]:
# =============================================================================
# MODEL FACTORY
# =============================================================================

def adapt_conv1_luminosity(old_conv: nn.Conv2d) -> nn.Conv2d:
    """
    Adapt 3-channel conv1 to 1-channel using luminosity weighting.
    Uses 0.299R + 0.587G + 0.114B — same as OASIS pipeline.
    """
    new_conv = nn.Conv2d(
        1, old_conv.out_channels,
        kernel_size = old_conv.kernel_size,
        stride      = old_conv.stride,
        padding     = old_conv.padding,
        bias        = False,
    )
    with torch.no_grad():
        w   = old_conv.weight.data.cpu()   # (out, 3, k, k)
        rgb = torch.tensor([0.299, 0.587, 0.114]).view(1, 3, 1, 1)
        new_conv.weight.data = (w * rgb).sum(dim=1, keepdim=True)
    return new_conv


def get_model(arch: str, init: str, num_classes: int) -> nn.Module:
    """
    Build model with 1-channel input and num_classes output.
    No partial freezing for ReID — full fine-tuning.

    Args:
        arch       : 'resnet18' or 'densenet121'
        init       : 'Pretrained' or 'Scratch'
        num_classes: number of patient identity classes
    """
    pretrained = (init.lower() == 'pretrained')

    if arch == 'resnet18':
        weights = models.ResNet18_Weights.IMAGENET1K_V1 \
                  if pretrained else None
        m       = models.resnet18(weights=weights)
        if pretrained:
            m.conv1 = adapt_conv1_luminosity(m.conv1)
        else:
            m.conv1 = nn.Conv2d(
                1, 64, kernel_size=7, stride=2,
                padding=3, bias=False
            )
            nn.init.kaiming_normal_(
                m.conv1.weight, mode='fan_out', nonlinearity='relu'
            )
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        nn.init.xavier_uniform_(m.fc.weight)
        nn.init.zeros_(m.fc.bias)
        return m.to(DEVICE)

    if arch == 'densenet121':
        weights = models.DenseNet121_Weights.IMAGENET1K_V1 \
                  if pretrained else None
        m       = models.densenet121(
            weights=weights, memory_efficient=True
        )
        old = m.features[0]   # conv0
        if pretrained:
            new = nn.Conv2d(
                1, old.out_channels,
                kernel_size = old.kernel_size,
                stride      = old.stride,
                padding     = old.padding,
                bias        = False,
            )
            with torch.no_grad():
                w   = old.weight.data.cpu()
                rgb = torch.tensor([0.299, 0.587, 0.114]).view(1,3,1,1)
                new.weight.data = (w * rgb).sum(dim=1, keepdim=True)
        else:
            new = nn.Conv2d(
                1, old.out_channels,
                kernel_size = old.kernel_size,
                stride      = old.stride,
                padding     = old.padding,
                bias        = False,
            )
            nn.init.kaiming_normal_(
                new.weight, mode='fan_out', nonlinearity='relu'
            )
        m.features[0]  = new
        m.classifier   = nn.Linear(
            m.classifier.in_features, num_classes
        )
        nn.init.xavier_uniform_(m.classifier.weight)
        nn.init.zeros_(m.classifier.bias)
        return m.to(DEVICE)

    raise ValueError(f"Unknown architecture: {arch}")


# ── Smoke test ────────────────────────────────────────────────────────────────
print("=== MODEL FACTORY SMOKE TEST ===")
for arch in ARCHITECTURES:
    for init in INITIALIZATIONS:
        m   = get_model(arch, init, NUM_CLASSES)
        d   = torch.zeros(2, 1, 224, 224).to(DEVICE)
        with torch.no_grad():
            out = m(d)
        total     = sum(p.numel() for p in m.parameters())
        trainable = sum(p.numel() for p in m.parameters()
                        if p.requires_grad)
        print(f"  {arch:<15} {init:<12} "
              f"output={list(out.shape)}  "
              f"params={total/1e6:.2f}M (all trainable for ReID)")
        del m, d, out
        torch.cuda.empty_cache()

print("\n✓ Model factory ready")


# =============================================================================
# EVALUATION METRICS
# =============================================================================

@torch.no_grad()
def evaluate_model(
    model  : nn.Module,
    loader : DataLoader,
) -> Dict[str, float]:
    """
    Evaluate model on a DataLoader.

    Returns:
        top1_acc         : Top-1 accuracy (%)
        top5_acc         : Top-5 accuracy (%)
        f1_macro         : Macro F1
        trueprob_top5_mean   : Mean P(true class) if true class in top-5
        trueprob_top5_median : Median of same
    """
    model.eval()
    all_preds  = []
    all_labels = []
    all_probs  = []

    for x, y in loader:
        x      = x.to(DEVICE, non_blocking=True)
        logits = model(x)
        probs  = torch.softmax(logits, dim=1)
        all_preds.append(logits.argmax(dim=1).cpu().numpy())
        all_labels.append(y.numpy())
        all_probs.append(probs.cpu().numpy())

    preds  = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    probs  = np.concatenate(all_probs)

    top1 = 100.0 * float(np.mean(preds == labels))

    k    = min(5, probs.shape[1])
    top5 = 100.0 * float(
        top_k_accuracy_score(labels, probs, k=k)
    ) if k > 1 else top1

    f1m = float(
        f1_score(labels, preds, average='macro', zero_division=0)
    )

    # TrueProbTop5: P(true class) if true class ranks in top-5, else 0
    true_prob_top5 = []
    for i in range(len(labels)):
        top_idx = np.argsort(probs[i])[-5:][::-1]
        if labels[i] in top_idx:
            true_prob_top5.append(float(probs[i, labels[i]]))
        else:
            true_prob_top5.append(0.0)
    true_prob_top5 = np.array(true_prob_top5)

    return dict(
        top1_acc              = top1,
        top5_acc              = top5,
        f1_macro              = f1m,
        trueprob_top5_mean    = float(true_prob_top5.mean()),
        trueprob_top5_median  = float(np.median(true_prob_top5)),
        trueprob_top5_std     = float(
            true_prob_top5.std(ddof=1)
        ) if len(true_prob_top5) > 1 else 0.0,
    )


def ci95(mean: float, std: float, n: int) -> Tuple[float, float]:
    if n <= 1:
        return mean, mean
    sem = std / math.sqrt(n)
    h   = 1.96 * sem
    return mean - h, mean + h


print("✓ Metrics functions ready")

=== MODEL FACTORY SMOKE TEST ===
  resnet18        Pretrained   output=[2, 1739]  params=12.06M (all trainable for ReID)
  resnet18        Scratch      output=[2, 1739]  params=12.06M (all trainable for ReID)
  densenet121     Pretrained   output=[2, 1739]  params=8.73M (all trainable for ReID)
  densenet121     Scratch      output=[2, 1739]  params=8.73M (all trainable for ReID)

✓ Model factory ready
✓ Metrics functions ready


In [4]:
# =============================================================================
# SAVING & RESUMABILITY HELPERS
# =============================================================================

def model_save_path(arch: str, init: str, gs_level: str) -> Path:
    d = MODELS_DIR / arch
    d.mkdir(parents=True, exist_ok=True)
    return d / f'{arch}_{init}_{gs_level}_best.pth'


def run_group_id(arch: str, init: str) -> str:
    return (
        f"NIH_CXR__{arch}_{init}__"
        f"NR{NUM_REPEATS}__E{NUM_EPOCHS}__"
        f"LR{LEARNING_RATE}__B{BATCH_SIZE}"
    )


def model_exists(arch: str, init: str, gs_level: str) -> bool:
    return model_save_path(arch, init, gs_level).exists()


def csv_row_exists(
    df       : pd.DataFrame,
    arch     : str,
    init     : str,
    gs_level : str,
) -> bool:
    if df.empty:
        return False
    return (
        (df['Model']          == arch)     &
        (df['Initialization'] == init)     &
        (df['GS_Level']       == gs_level)
    ).any()


def save_results_csv(rows: List[Dict]) -> Path:
    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_CSV, index=False)
    return RESULTS_CSV


print("✓ Saving helpers ready")

✓ Saving helpers ready


In [5]:
# =============================================================================
# TRAINING ENGINE
# =============================================================================

def train_one_repeat(
    arch      : str,
    init      : str,
    gs_level  : str,
    repeat_idx: int,
    x_train   : np.ndarray,
    y_train   : np.ndarray,
    x_val     : np.ndarray,
    y_val     : np.ndarray,
    x_test    : np.ndarray,
    y_test    : np.ndarray,
    mean      : float,
    std       : float,
) -> Tuple[Dict[str, float], nn.Module]:
    """
    One full train/val/test run for a single repeat.

    Returns:
        metrics : dict of evaluation metrics on test set
        model   : trained model (best checkpoint)
    """
    repeat_seed = GLOBAL_SEED + 1000 * repeat_idx
    set_global_seed(repeat_seed)

    dl_gen = torch.Generator()
    dl_gen.manual_seed(repeat_seed)

    model = get_model(arch, init, NUM_CLASSES)

    train_ds = ReIDDataset(x_train, y_train, mean, std, augment=True)
    val_ds   = ReIDDataset(x_val,   y_val,   mean, std, augment=False)
    test_ds  = ReIDDataset(x_test,  y_test,  mean, std, augment=False)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=4, pin_memory=True,
        worker_init_fn=seed_worker, generator=dl_gen
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=4, pin_memory=True,
    )
    test_loader = DataLoader(
        test_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=4, pin_memory=True,
    )

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE, weight_decay=1e-4
    )

    # Cosine annealing with 10% linear warmup
    steps_per_epoch = max(1, len(train_loader))
    total_steps     = NUM_EPOCHS * steps_per_epoch
    warmup_steps    = int(0.10 * total_steps)

    def lr_lambda(step: int) -> float:
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(
            max(1, total_steps - warmup_steps)
        )
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lr_lambda
    )

    best_val_acc = -1.0
    best_state   = {
        k: v.detach().cpu().clone()
        for k, v in model.state_dict().items()
    }
    patience     = 8
    patience_ctr = 0

    use_amp = (DEVICE.type == 'cuda')
    scaler  = torch.cuda.amp.GradScaler(enabled=use_amp)

    for epoch in range(NUM_EPOCHS):
        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(
                device_type='cuda', enabled=use_amp
            ):
                logits = model(xb)
                loss   = criterion(logits, yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), 1.0
            )
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

        # ── Validate ──────────────────────────────────────────────────────────
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb   = xb.to(DEVICE, non_blocking=True)
                pred = model(xb).argmax(dim=1).cpu()
                correct += int((pred == yb).sum().item())
                total   += int(yb.numel())
        val_acc = 100.0 * correct / max(total, 1)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            patience_ctr = 0
        else:
            patience_ctr += 1

        if patience_ctr >= patience:
            break

    model.load_state_dict(best_state)
    metrics = evaluate_model(model, test_loader)
    return metrics, model


print("✓ Training engine ready")

✓ Training engine ready


## Main Experiment Loop

In [6]:
# =============================================================================
# MAIN EXPERIMENT LOOP
# =============================================================================

# Load existing results for resumability
if RESUME and RESULTS_CSV.exists():
    df_existing  = pd.read_csv(RESULTS_CSV)
    all_rows     = df_existing.to_dict('records')
    print(f"🔄 Loaded {len(all_rows)} existing result rows")
else:
    df_existing = pd.DataFrame()
    all_rows    = []
    print("🆕 No existing results — starting fresh")

print(f"\n{'='*80}")
print(f"🚀 NIH CXR ReID — SINGLE SPLIT + {NUM_REPEATS} REPEATS")
print(f"   Time: {now_str()}")
print(f"   Classes: {NUM_CLASSES:,} patients")
print(f"   Random Rank-1 baseline: {100/NUM_CLASSES:.3f}%")
print(f"{'='*80}")

for arch in ARCHITECTURES:
    for init in INITIALIZATIONS:

        rgid = run_group_id(arch, init)
        raw_perrun_top1 = None  # for paired stats vs raw

        print(f"\n{'█'*80}")
        print(f"  {arch} | {init} | {rgid}")
        print(f"{'█'*80}")

        for gs_level in GS_LEVELS:

            # ── Resume guard ──────────────────────────────────────────────────
            if RESUME and csv_row_exists(df_existing, arch, init, gs_level):
                # Load raw_perrun_top1 for stats continuity
                if gs_level == 'raw' and raw_perrun_top1 is None:
                    row = df_existing[
                        (df_existing['Model']          == arch) &
                        (df_existing['Initialization'] == init) &
                        (df_existing['GS_Level']       == 'raw')
                    ].iloc[0]
                    try:
                        raw_perrun_top1 = eval(row['PerRun_Top1'])
                    except Exception:
                        raw_perrun_top1 = None
                print(f"⏩  Skip: {arch} | {init} | {gs_level} "
                      f"(already computed)")
                continue

            print(f"\n{'─'*60}")
            print(f"  Condition : {gs_level}")
            print(f"{'─'*60}")

            # ── Load data for this condition ──────────────────────────────────
            x_train, y_train, _, _ = load_reid_npz('train', gs_level)
            x_val,   y_val,   _, _ = load_reid_npz('val',   gs_level)
            x_test,  y_test,  _, _ = load_reid_npz('test',  gs_level)

            # Per-condition normalisation (train+val pool)
            mean, std = compute_mean_std(x_train, x_val)
            print(f"  Normalisation: mean={mean:.4f}  std={std:.4f}")

            perrun_top1  = []
            perrun_top5  = []
            perrun_f1    = []
            perrun_tp5m  = []
            perrun_tp5md = []
            saved_model  = None

            model_already_exists = (
                RESUME and model_exists(arch, init, gs_level)
            )

            # ── Repeats ───────────────────────────────────────────────────────
            for r in range(NUM_REPEATS):

                if model_already_exists:
                    if r == 0:
                        print(f"  ✓ Model exists → eval only")
                    # Load model and evaluate
                    m = get_model(arch, init, NUM_CLASSES)
                    ckpt = torch.load(
                        model_save_path(arch, init, gs_level),
                        map_location=DEVICE,
                        weights_only=True
                    )
                    m.load_state_dict(ckpt['state_dict'])
                    test_ds = ReIDDataset(
                        x_test, y_test, mean, std, augment=False
                    )
                    test_loader = DataLoader(
                        test_ds, batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=4
                    )
                    metrics = evaluate_model(m, test_loader)
                    del m
                    torch.cuda.empty_cache()
                else:
                    metrics, model_obj = train_one_repeat(
                        arch       = arch,
                        init       = init,
                        gs_level   = gs_level,
                        repeat_idx = r,
                        x_train    = x_train,
                        y_train    = y_train,
                        x_val      = x_val,
                        y_val      = y_val,
                        x_test     = x_test,
                        y_test     = y_test,
                        mean       = mean,
                        std        = std,
                    )
                    if r == 0:
                        saved_model = model_obj

                perrun_top1.append(float(metrics['top1_acc']))
                perrun_top5.append(float(metrics['top5_acc']))
                perrun_f1.append(float(metrics['f1_macro']))
                perrun_tp5m.append(
                    float(metrics['trueprob_top5_mean'])
                )
                perrun_tp5md.append(
                    float(metrics['trueprob_top5_median'])
                )

                print(
                    f"  Repeat {r+1}/{NUM_REPEATS}: "
                    f"Top1={metrics['top1_acc']:.2f}%  "
                    f"Top5={metrics['top5_acc']:.2f}%  "
                    f"F1={metrics['f1_macro']:.4f}"
                )

            # ── Save model (repeat 0 only) ────────────────────────────────────
            if saved_model is not None:
                sp = model_save_path(arch, init, gs_level)
                torch.save(
                    {
                        'arch'       : arch,
                        'init'       : init,
                        'gs_level'   : gs_level,
                        'num_classes': NUM_CLASSES,
                        'state_dict' : saved_model.state_dict(),
                        'top1_mean'  : float(np.mean(perrun_top1)),
                        'perrun_top1': perrun_top1,
                    },
                    sp
                )
                print(f"  💾 Saved: {sp.name}")
                del saved_model
                torch.cuda.empty_cache()

            if gs_level == 'raw':
                raw_perrun_top1 = perrun_top1[:]

            # ── Aggregate ─────────────────────────────────────────────────────
            top1_mean = float(np.mean(perrun_top1))
            top1_std  = float(
                np.std(perrun_top1, ddof=1)
            ) if len(perrun_top1) > 1 else 0.0
            top1_lo, top1_hi = ci95(top1_mean, top1_std, len(perrun_top1))

            top5_mean = float(np.mean(perrun_top5))
            top5_std  = float(
                np.std(perrun_top5, ddof=1)
            ) if len(perrun_top5) > 1 else 0.0

            f1_mean = float(np.mean(perrun_f1))
            f1_std  = float(
                np.std(perrun_f1, ddof=1)
            ) if len(perrun_f1) > 1 else 0.0

            tp5_mean = float(np.mean(perrun_tp5m))
            tp5_med  = float(np.median(perrun_tp5md))

            # ── Accuracy drop vs raw ──────────────────────────────────────────
            acc_drop_pp  = ''
            acc_drop_rel = ''
            p_value      = ''

            if raw_perrun_top1 is not None and gs_level != 'raw':
                acc_drop_pp  = float(
                    np.mean(raw_perrun_top1) - top1_mean
                )
                acc_drop_rel = float(
                    acc_drop_pp /
                    (np.mean(raw_perrun_top1) + 1e-8) * 100.0
                )
                if len(raw_perrun_top1) == len(perrun_top1):
                    _, p_value = stats.ttest_rel(
                        raw_perrun_top1, perrun_top1
                    )

            # ── Build blind eval (GS trained, raw test) ───────────────────────
            blind_top1 = blind_top5 = blind_f1 = ''
            blind_tp5m = blind_tp5md = ''

            if gs_level != 'raw':
                # Load raw test and evaluate saved model
                x_test_raw, y_test_raw, _, _ = load_reid_npz(
                    'test', 'raw'
                )
                mean_raw, std_raw = compute_mean_std(
                    *[load_reid_npz('train','raw')[:2],
                      load_reid_npz('val',  'raw')[:2]]
                ) if False else (mean, std)

                # Use same mean/std as GS condition for fair comparison
                m_blind = get_model(arch, init, NUM_CLASSES)
                ckpt    = torch.load(
                    model_save_path(arch, init, gs_level),
                    map_location=DEVICE, weights_only=True
                )
                m_blind.load_state_dict(ckpt['state_dict'])
                raw_test_ds     = ReIDDataset(
                    x_test_raw, y_test_raw, mean, std, augment=False
                )
                raw_test_loader = DataLoader(
                    raw_test_ds, batch_size=BATCH_SIZE,
                    shuffle=False, num_workers=4
                )
                blind_metrics = evaluate_model(m_blind, raw_test_loader)
                del m_blind, raw_test_ds, x_test_raw
                torch.cuda.empty_cache()

                blind_top1  = float(blind_metrics['top1_acc'])
                blind_top5  = float(blind_metrics['top5_acc'])
                blind_f1    = float(blind_metrics['f1_macro'])
                blind_tp5m  = float(blind_metrics['trueprob_top5_mean'])
                blind_tp5md = float(
                    blind_metrics['trueprob_top5_median']
                )

            # ── Log row ───────────────────────────────────────────────────────
            row = dict(
                RunGroupID             = rgid,
                Model                  = arch,
                Initialization         = init,
                GS_Level               = gs_level,
                GS_Percentage          = float(
                    GS_PERCENTAGE_MAP[gs_level]
                ),
                Top1_Mean              = top1_mean,
                Top1_Std               = top1_std,
                Top1_CI_Lower          = float(top1_lo),
                Top1_CI_Upper          = float(top1_hi),
                Top5_Mean              = top5_mean,
                Top5_Std               = top5_std,
                F1_Mean                = f1_mean,
                F1_Std                 = f1_std,
                TrueProbTop5_Mean      = tp5_mean,
                TrueProbTop5_Median    = tp5_med,
                Top1_GSTrain_RawTest   = blind_top1,
                Top5_GSTrain_RawTest   = blind_top5,
                F1_GSTrain_RawTest     = blind_f1,
                TrueProb_GSTrain_RawTest_Mean   = blind_tp5m,
                TrueProb_GSTrain_RawTest_Median = blind_tp5md,
                AccuracyDrop_pp        = acc_drop_pp,
                AccuracyDrop_relative  = acc_drop_rel,
                pvalue_vs_raw_paired   = p_value,
                N_Repeats              = int(NUM_REPEATS),
                PerRun_Top1            = str(perrun_top1),
            )

            all_rows.append(row)
            save_results_csv(all_rows)
            print(
                f"\n  ── {gs_level} SUMMARY ──\n"
                f"  Top1={top1_mean:.2f}%±{top1_std:.2f}  "
                f"[{top1_lo:.2f},{top1_hi:.2f}]  "
                f"Top5={top5_mean:.2f}%  "
                f"F1={f1_mean:.4f}"
            )
            if gs_level != 'raw' and acc_drop_pp != '':
                print(
                    f"  Drop={acc_drop_pp:.2f}pp "
                    f"({acc_drop_rel:.1f}%)  "
                    f"p={p_value:.4f}  "
                    f"Blind Top1={blind_top1:.2f}%"
                )

            # Clean up condition data
            del x_train, x_val, x_test
            gc.collect()
            torch.cuda.empty_cache()

# ── Final save ────────────────────────────────────────────────────────────────
df_final = save_results_csv(all_rows)
print(f"\n{'='*80}")
print(f"✅ ALL EXPERIMENTS COMPLETE")
print(f"   Results: {RESULTS_CSV}")
print(f"{'='*80}")
pd.read_csv(RESULTS_CSV)

🆕 No existing results — starting fresh

🚀 NIH CXR ReID — SINGLE SPLIT + 5 REPEATS
   Time: 2026-04-12 08:40:08
   Classes: 1,739 patients
   Random Rank-1 baseline: 0.058%

████████████████████████████████████████████████████████████████████████████████
  resnet18 | Pretrained | NIH_CXR__resnet18_Pretrained__NR5__E30__LR0.0005__B64
████████████████████████████████████████████████████████████████████████████████

────────────────────────────────────────────────────────────
  Condition : raw
────────────────────────────────────────────────────────────
  Normalisation: mean=0.4950  std=0.2445
  Repeat 1/5: Top1=69.37%  Top5=81.54%  F1=0.6526
  Repeat 2/5: Top1=69.63%  Top5=81.57%  F1=0.6564
  Repeat 3/5: Top1=69.63%  Top5=82.58%  F1=0.6570
  Repeat 4/5: Top1=69.70%  Top5=81.73%  F1=0.6595
  Repeat 5/5: Top1=69.11%  Top5=81.18%  F1=0.6519
  💾 Saved: resnet18_Pretrained_raw_best.pth

  ── raw SUMMARY ──
  Top1=69.49%±0.24  [69.27,69.70]  Top5=81.72%  F1=0.6555

─────────────────────────────

KeyboardInterrupt: 

In [ ]:
print('done')

## Results

In [ ]:
# =============================================================================
# RESULTS VISUALISATION
# =============================================================================

df = pd.read_csv(RESULTS_CSV)

# Clean numeric columns
for col in ['Top1_Mean', 'Top5_Mean', 'Top1_Std', 'Top5_Std',
            'AccuracyDrop_pp', 'AccuracyDrop_relative',
            'Top1_GSTrain_RawTest', 'Top5_GSTrain_RawTest']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

COND_ORDER  = ['raw', 'gs50', 'gs40', 'gs30', 'gs20', 'gs10', 'gs0']
COND_LABELS = ['Raw','GS-50','GS-40','GS-30','GS-20','GS-10','GS-0']
COND_XVALS  = {c: i for i, c in enumerate(COND_ORDER)}
ARCH_COLORS = {'resnet18': '#4C72B0', 'densenet121': '#DD8452'}
INIT_LINES  = {'Pretrained': '-', 'Scratch': '--'}
RAND_BASELINE = 100.0 / NUM_CLASSES

df['x'] = df['GS_Level'].map(COND_XVALS)

# ── Figure 1: Top-1 Accuracy with CI ribbons ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7), sharey=True)
fig.suptitle(
    'NIH CXR ReID — Top-1 Accuracy vs GS Condition\n'
    f'1,739-class patient identification  '
    f'(random baseline={RAND_BASELINE:.3f}%)',
    fontsize=14, fontweight='bold'
)

for ax_idx, init in enumerate(['Pretrained', 'Scratch']):
    ax  = axes[ax_idx]
    sub = df[df['Initialization'] == init].copy()

    for arch in ARCHITECTURES:
        a = sub[sub['Model'] == arch].sort_values('x')
        if a.empty:
            continue
        xs   = a['x'].values
        y    = a['Top1_Mean'].values
        yerr = a['Top1_Std'].values

        ax.plot(
            xs, y, marker='o', linewidth=2.5, markersize=8,
            linestyle=INIT_LINES[init],
            color=ARCH_COLORS[arch], label=arch
        )
        ax.fill_between(
            xs, y - yerr, y + yerr,
            alpha=0.15, color=ARCH_COLORS[arch]
        )
        for xi, yi in zip(xs, y):
            ax.annotate(
                f'{yi:.1f}%', (xi, yi),
                textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=7.5
            )

    ax.axhline(
        RAND_BASELINE, color='red', linestyle=':',
        linewidth=1.5, label=f'Random ({RAND_BASELINE:.3f}%)'
    )
    ax.set_title(f'{init}', fontsize=12, fontweight='bold')
    ax.set_xticks(range(len(COND_ORDER)))
    ax.set_xticklabels(COND_LABELS, fontsize=9)
    ax.set_ylabel('Top-1 Accuracy (%)')
    ax.set_xlabel('Condition')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig1_top1_accuracy.png', dpi=150,
            bbox_inches='tight')
plt.show()
print("Saved → fig1_top1_accuracy.png")


# ── Figure 2: Accuracy drop ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle(
    'Accuracy Drop vs Raw Baseline (pp)\n'
    'Adaptive condition | Larger = more impact from GS',
    fontsize=14, fontweight='bold'
)

gs_only = df[df['GS_Level'] != 'raw'].copy()

for ax_idx, init in enumerate(['Pretrained', 'Scratch']):
    ax  = axes[ax_idx]
    sub = gs_only[gs_only['Initialization'] == init]

    for arch in ARCHITECTURES:
        a = sub[sub['Model'] == arch].sort_values('x')
        if a.empty:
            continue
        ax.bar(
            a['x'] + (0.2 if arch == 'densenet121' else -0.2),
            a['AccuracyDrop_pp'],
            width=0.35,
            color=ARCH_COLORS[arch], alpha=0.85, label=arch
        )

    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'{init}', fontsize=11, fontweight='bold')
    ax.set_xticks(range(len(COND_ORDER)))
    ax.set_xticklabels(COND_LABELS, fontsize=9)
    ax.set_xlabel('Condition')
    ax.set_ylabel('Accuracy Drop (pp)')
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig2_accuracy_drop.png', dpi=150,
            bbox_inches='tight')
plt.show()
print("Saved → fig2_accuracy_drop.png")


# ── Figure 3: Adaptive vs Blind ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)
fig.suptitle(
    'Adaptive vs Blind ReID Accuracy\n'
    'Adaptive: test on same GS condition  |  '
    'Blind: trained on GS, tested on raw',
    fontsize=14, fontweight='bold'
)

for ax_idx, init in enumerate(['Pretrained', 'Scratch']):
    ax  = axes[ax_idx]
    sub = df[df['Initialization'] == init].copy()

    for arch in ARCHITECTURES:
        a = sub[sub['Model'] == arch].sort_values('x')
        if a.empty:
            continue
        xs = a['x'].values

        # Adaptive
        ax.plot(
            xs, a['Top1_Mean'].values,
            marker='o', linewidth=2, color=ARCH_COLORS[arch],
            linestyle='-', label=f'{arch} Adaptive'
        )
        # Blind (GS only)
        blind_vals = pd.to_numeric(
            a['Top1_GSTrain_RawTest'], errors='coerce'
        ).values
        ax.plot(
            xs[1:], blind_vals[1:],   # skip raw (no blind for raw)
            marker='s', linewidth=2, color=ARCH_COLORS[arch],
            linestyle='--', label=f'{arch} Blind', alpha=0.7
        )

    ax.axhline(
        RAND_BASELINE, color='red', linestyle=':',
        linewidth=1.5, alpha=0.7
    )
    ax.set_title(f'{init}', fontsize=11, fontweight='bold')
    ax.set_xticks(range(len(COND_ORDER)))
    ax.set_xticklabels(COND_LABELS, fontsize=9)
    ax.set_xlabel('Training Condition')
    ax.set_ylabel('Top-1 Accuracy (%)')
    ax.legend(fontsize=8, ncol=2)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3_adaptive_vs_blind.png', dpi=150,
            bbox_inches='tight')
plt.show()
print("Saved → fig3_adaptive_vs_blind.png")


# ── Figure 4: Top-1 vs Top-5 comparison ──────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle(
    'Top-1 vs Top-5 Accuracy across GS Conditions',
    fontsize=14, fontweight='bold'
)

for row_idx, init in enumerate(['Pretrained', 'Scratch']):
    for col_idx, metric in enumerate(['Top1_Mean', 'Top5_Mean']):
        ax    = axes[row_idx][col_idx]
        label = 'Top-1' if metric == 'Top1_Mean' else 'Top-5'
        sub   = df[df['Initialization'] == init].copy()

        for arch in ARCHITECTURES:
            a = sub[sub['Model'] == arch].sort_values('x')
            if a.empty:
                continue
            ax.plot(
                a['x'], a[metric],
                marker='o', linewidth=2, markersize=7,
                color=ARCH_COLORS[arch], label=arch
            )

        ax.axhline(
            RAND_BASELINE, color='red', linestyle=':',
            linewidth=1.2, label='Random'
        )
        ax.set_title(
            f'{label} — {init}',
            fontsize=11, fontweight='bold'
        )
        ax.set_xticks(range(len(COND_ORDER)))
        ax.set_xticklabels(COND_LABELS, fontsize=9)
        ax.set_xlabel('Condition')
        ax.set_ylabel(f'{label} Accuracy (%)')
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig4_top1_vs_top5.png', dpi=150,
            bbox_inches='tight')
plt.show()
print("Saved → fig4_top1_vs_top5.png")


# ── Figure 5: Pretrained vs Scratch gap ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
fig.suptitle(
    'Pretrained vs Scratch Gap (Top-1)\n'
    'Positive = Pretrained better',
    fontsize=14, fontweight='bold'
)

for arch in ARCHITECTURES:
    pre = df[
        (df['Model'] == arch) &
        (df['Initialization'] == 'Pretrained')
    ].sort_values('x')
    scr = df[
        (df['Model'] == arch) &
        (df['Initialization'] == 'Scratch')
    ].sort_values('x')
    m = pre[['x', 'GS_Level', 'Top1_Mean']].merge(
        scr[['x', 'GS_Level', 'Top1_Mean']],
        on=['x', 'GS_Level'], suffixes=('_pre', '_scr')
    )
    if m.empty:
        continue
    gap = m['Top1_Mean_pre'] - m['Top1_Mean_scr']
    ax.plot(
        m['x'], gap, marker='o', linewidth=2.5,
        color=ARCH_COLORS[arch], label=arch
    )
    ax.fill_between(m['x'], gap, 0, alpha=0.08,
                    color=ARCH_COLORS[arch])

ax.axhline(0, color='black', linewidth=1, linestyle='--')
ax.set_xticks(range(len(COND_ORDER)))
ax.set_xticklabels(COND_LABELS, fontsize=9)
ax.set_xlabel('Condition')
ax.set_ylabel('Top-1 Gap (pp)')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig5_pretrained_vs_scratch.png', dpi=150,
            bbox_inches='tight')
plt.show()
print("Saved → fig5_pretrained_vs_scratch.png")


# ── Figure 6: Summary table ───────────────────────────────────────────────────
summary_cols = [
    'Model', 'Initialization', 'GS_Level',
    'Top1_Mean', 'Top1_Std', 'Top1_CI_Lower', 'Top1_CI_Upper',
    'Top5_Mean', 'AccuracyDrop_pp', 'AccuracyDrop_relative',
    'pvalue_vs_raw_paired', 'Top1_GSTrain_RawTest'
]
available = [c for c in summary_cols if c in df.columns]
summary   = df[available].round(3)

print("\n=== RESULTS SUMMARY TABLE ===")
print(summary.to_string(index=False))

print(f"\n=== ALL FIGURES SAVED TO {FIG_DIR} ===")
for f in sorted(FIG_DIR.iterdir()):
    print(f"  {f.name}")